# Tuned Linear SVM on TF-IDF

This notebook tunes `LinearSVC` and evaluates the best model on a held-out test set.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import LinearSVC

In [6]:
from pathlib import Path
cwd = Path.cwd().resolve()

# Search upward so this notebook works from project root, notebooks/, or notebooks/experiments/.
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists() and (candidate / "requirements.txt").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not resolve project root from {cwd}. "
        "Open the notebook from within the hate-speech-detection workspace."
    )

TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
X_PATH = TFIDF_DIR / "X_tfidf.npz"
Y_PATH = TFIDF_DIR / "y_labels.csv"
if not X_PATH.exists() or not Y_PATH.exists():
    raise FileNotFoundError(
        "Missing TF-IDF artifacts. Run these first:\n"
        "1) .\\.venv\\Scripts\\python.exe scripts/preprocess_data.py\n"
        "2) .\\.venv\\Scripts\\python.exe scripts/vectorize_tfidf.py"
    )

In [7]:
X = load_npz(X_PATH)
y = pd.read_csv(Y_PATH)["class"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X shape: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Class distribution:")
print(pd.Series(y).value_counts().sort_index())

X shape: (24783, 20000)
Train: (19826, 20000), Test: (4957, 20000)
Class distribution:
0     1430
1    19190
2     4163
Name: count, dtype: int64


In [8]:
baseline_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    loss="squared_hinge",
    random_state=42,
)
baseline_svm.fit(X_train, y_train)
baseline_pred = baseline_svm.predict(X_test)

print("Baseline LinearSVC metrics:")
print(f"accuracy: {accuracy_score(y_test, baseline_pred):.4f}")
print(f"f1_macro: {f1_score(y_test, baseline_pred, average='macro', zero_division=0):.4f}")
print(f"class0_f1: {classification_report(y_test, baseline_pred, output_dict=True, zero_division=0)['0']['f1-score']:.4f}")

Baseline LinearSVC metrics:
accuracy: 0.8927
f1_macro: 0.7319
class0_f1: 0.4075


In [9]:
param_grid = {
    "C": [0.1, 0.5, 1.0, 2.0, 5.0],
    "class_weight": ["balanced", {0: 3, 1: 1, 2: 1.5}, {0: 4, 1: 1, 2: 1.5}],
    "loss": ["hinge", "squared_hinge"],
}

grid = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=1,
    verbose=1,
)

grid.fit(X_train, y_train)
best_svm = grid.best_estimator_

print("Best params:", grid.best_params_)
print(f"Best CV f1_macro: {grid.best_score_:.4f}")

Fitting 3 folds for each of 30 candidates, totalling 90 fits


c:\Users\Atharva\OneDrive\Desktop\WebD\hate-speech-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Atharva\OneDrive\Desktop\WebD\hate-speech-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Atharva\OneDrive\Desktop\WebD\hate-speech-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Atharva\OneDrive\Desktop\WebD\hate-speech-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Atharva\OneDrive\Desktop\WebD\hate-speech-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear fa

Best params: {'C': 0.5, 'class_weight': 'balanced', 'loss': 'hinge'}
Best CV f1_macro: 0.7421


In [10]:
tuned_pred = best_svm.predict(X_test)

metrics_df = pd.DataFrame([
    {
        "model": "LinearSVC (baseline)",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "precision_macro": precision_score(y_test, baseline_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, baseline_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, baseline_pred, average="weighted", zero_division=0),
        "class0_f1": classification_report(y_test, baseline_pred, output_dict=True, zero_division=0)["0"]["f1-score"],
        "class0_recall": classification_report(y_test, baseline_pred, output_dict=True, zero_division=0)["0"]["recall"],
    },
    {
        "model": "LinearSVC (tuned)",
        "accuracy": accuracy_score(y_test, tuned_pred),
        "precision_macro": precision_score(y_test, tuned_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, tuned_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, tuned_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, tuned_pred, average="weighted", zero_division=0),
        "class0_f1": classification_report(y_test, tuned_pred, output_dict=True, zero_division=0)["0"]["f1-score"],
        "class0_recall": classification_report(y_test, tuned_pred, output_dict=True, zero_division=0)["0"]["recall"],
    },
]).set_index("model")

metrics_df.round(4)

,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,class0_f1,class0_recall
model,,,,,,,
LinearSVC (baseline),0.8927,0.7338,0.7319,0.7319,0.8912,0.4075,0.3811
LinearSVC (tuned),0.8947,0.7380,0.7693,0.7512,0.8954,0.4624,0.4510


In [11]:
print("Tuned LinearSVC classification report:\n")
print(classification_report(y_test, tuned_pred, digits=4, zero_division=0))

labels = np.sort(np.unique(y))
cm = confusion_matrix(y_test, tuned_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

Tuned LinearSVC classification report:

              precision    recall  f1-score   support

           0     0.4743    0.4510    0.4624       286
           1     0.9555    0.9182    0.9365      3838
           2     0.7844    0.9388    0.8546       833

    accuracy                         0.8947      4957
   macro avg     0.7380    0.7693    0.7512      4957
weighted avg     0.8990    0.8947    0.8954      4957



,pred_0,pred_1,pred_2
true_0,129,122,35
true_1,134,3524,180
true_2,9,42,782
